# 🧠 Notebook 5: Word Embeddings y LSTM

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ ¿Qué son **Word Embeddings**?
2. ✅ Diferencia entre **TF-IDF** y **Embeddings**
3. ✅ Construir modelo **LSTM** para texto
4. ✅ Por qué LSTM entiende **"not good"** (orden importa)
5. ✅ Entrenar y evaluar red neuronal

⏱️ **Tiempo estimado**: 35 minutos (incluye entrenamiento)

---

## 💡 Problema: TF-IDF No Entiende el Orden

### ❌ Con TF-IDF:

```python
"not good" = ["not", "good"]  → Vector TF-IDF
"good" = ["good"]              → Vector TF-IDF similar
```

**Problema**: TF-IDF solo cuenta palabras, NO entiende el orden.

### ✅ Con LSTM:

```python
"not" → estado_1
"good" + estado_1 → estado_2 (negativo!)
```

**Solución**: LSTM procesa secuencialmente y "recuerda" contexto.

---

## 🔧 Setup Inicial

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import data_loader, deep_model
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

print("✅ Setup completo!")

## 📥 Cargar y Preparar Datos

In [ ]:
# Cargar dataset (usamos subset para que sea más rápido)
print("📥 Cargando dataset IMDB...")
(X_train_raw, y_train), (X_test_raw, y_test) = data_loader.load_imdb_data(num_words=10000)

# Usar subset
n_samples = 5000
X_train_raw = X_train_raw[:n_samples]
y_train = y_train[:n_samples]
X_test_raw = X_test_raw[:1000]  # 1000 para test
y_test = y_test[:1000]

print(f"\n✅ Datos cargados:")
print(f"   • Training: {len(X_train_raw)} reviews")
print(f"   • Test: {len(X_test_raw)} reviews")

## 🔢 Padding: Hacer Todas las Secuencias del Mismo Tamaño

LSTM necesita que todas las secuencias tengan la **misma longitud**:

In [ ]:
# Aplicar padding
max_length = 200  # Longitud máxima

print(f"🔧 Aplicando padding (longitud={max_length})...")
X_train_padded = data_loader.pad_sequences(X_train_raw, maxlen=max_length)
X_test_padded = data_loader.pad_sequences(X_test_raw, maxlen=max_length)

print(f"\n✅ Padding completado:")
print(f"   • Shape antes: lista de listas de longitud variable")
print(f"   • Shape después: {X_train_padded.shape}")

# Ejemplo
print(f"\n📝 Ejemplo de padding:")
print(f"   Longitud original: {len(X_train_raw[0])} palabras")
print(f"   Secuencia paddeada: {X_train_padded[0][:20]}... (primeras 20)")
print(f"   💡 Los ceros al inicio son el padding")

## 🏗️ Construir Modelo LSTM

### Arquitectura:

```
1. Embedding Layer: Índices → Vectores densos
2. LSTM Layer: Procesa secuencia
3. Dense Layer: Clasificación
4. Output: Probabilidad (0-1)
```

In [ ]:
# Parámetros
vocab_size = 10000  # Top 10k palabras
embedding_dim = 100  # Dimensión de embeddings

# Construir modelo
print("🏗️ Construyendo modelo LSTM...")

model = Sequential([
    # 1. Embedding: Convierte índices → vectores
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
    
    # 2. LSTM bidireccional
    Bidirectional(LSTM(64, return_sequences=False)),
    
    # 3. Dropout (prevenir overfitting)
    Dropout(0.5),
    
    # 4. Dense
    Dense(32, activation='relu'),
    
    # 5. Output
    Dense(1, activation='sigmoid')  # Probabilidad 0-1
])

# Compilar
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n✅ Modelo construido!")
print("\n📊 Arquitectura:")
model.summary()

## 🎓 Entrenar Modelo

**Nota**: Esto puede tardar 5-10 minutos dependiendo de tu hardware.

In [ ]:
# Split para validación
X_train, X_val, y_train_split, y_val = train_test_split(
    X_train_padded, y_train, 
    test_size=0.2, 
    random_state=42
)

print(f"📊 División de datos:")
print(f"   • Training: {X_train.shape[0]} samples")
print(f"   • Validation: {X_val.shape[0]} samples")
print(f"   • Test: {X_test_padded.shape[0]} samples")

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Entrenar
print("\n🚀 Entrenando modelo LSTM...")
print("   (Esto puede tardar 5-10 minutos)\n")

history = model.fit(
    X_train, y_train_split,
    validation_data=(X_val, y_val),
    epochs=5,  # Pocas épocas para demo
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

print("\n✅ Entrenamiento completado!")

## 📈 Visualizar Training History

In [ ]:
# Crear gráficos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Época', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Pérdida Durante Entrenamiento', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_xlabel('Época', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Precisión Durante Entrenamiento', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Lo que buscamos:")
print("   • Loss bajando ✅")
print("   • Accuracy subiendo ✅")
print("   • Val_loss cerca de loss (no overfitting) ✅")

## 📊 Evaluar en Test Set

In [ ]:
# Evaluar
print("📊 Evaluando en test set...")
test_loss, test_acc = model.evaluate(X_test_padded, y_test, verbose=0)

print(f"\n✅ RESULTADOS EN TEST SET:")
print("="*50)
print(f"   • Test Loss: {test_loss:.4f}")
print(f"   • Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print("="*50)

## 🎯 Ejemplo: ¿Por Qué LSTM Entiende "not good"?

Veamos cómo LSTM procesa secuencias:

In [ ]:
# Obtener diccionario de palabras
word_index = data_loader.get_word_index()

# Ejemplos para probar
test_examples = [
    "good movie",           # Positivo
    "not good movie",       # Negativo (orden importa!)
    "excellent film",       # Positivo
    "terrible waste",       # Negativo
]

print("🧪 PROBANDO COMPRENSIÓN DE ORDEN:")
print("="*70)

for text in test_examples:
    # Convertir a secuencia
    sequence = [word_index.get(word.lower(), 2) for word in text.split()]
    
    # Padding
    padded = keras.preprocessing.sequence.pad_sequences(
        [sequence], 
        maxlen=max_length, 
        padding='pre'
    )
    
    # Predecir
    prob = model.predict(padded, verbose=0)[0][0]
    sentiment = "Positivo ✅" if prob >= 0.5 else "Negativo ❌"
    
    print(f"\nTexto: '{text}'")
    print(f"   Probabilidad: {prob:.4f}")
    print(f"   Predicción: {sentiment}")

print("\n" + "="*70)
print("💡 Observa cómo 'good movie' es positivo")
print("   pero 'not good movie' es negativo")
print("   ¡LSTM entiende el ORDEN de las palabras!")

## 📊 Resumen

En este notebook aprendiste:

✅ **Word Embeddings**: Representación vectorial densa de palabras
✅ **LSTM**: Procesa secuencias entendiendo orden y contexto
✅ **Bidirectional LSTM**: Lee en ambas direcciones
✅ **Padding**: Hacer secuencias del mismo tamaño
✅ **Ventaja sobre TF-IDF**: Entiende "not good" ≠ "good"

---

## 🎓 Próximo Paso

En el **Notebook 6** aprenderás:
- 🔮 Usar modelos entrenados para predicciones
- 🎯 Predecir tus propias reviews
- 📊 Comparar diferentes modelos

**¡Nos vemos en el siguiente notebook!** 🚀